# Forecast Backtesting

Forecast evaluation must preserve temporal order. This notebook performs expanding-window one-step-ahead backtesting and compares a naive forecast with an AR(1) model.


In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.ar_model import AutoReg

rng = np.random.default_rng(42)
n = 180
y = np.zeros(n)
for t in range(1, n):
    y[t] = 0.7 * y[t-1] + rng.normal()
initial_window = 80
actual, naive, ar1 = [], [], []
for origin in range(initial_window, n):
    train = y[:origin]
    actual.append(y[origin])
    naive.append(train[-1])
    model = AutoReg(train, lags=1, trend='c', old_names=False).fit()
    ar1.append(model.predict(start=origin, end=origin)[0])
actual = np.asarray(actual)
scale = np.mean(np.abs(np.diff(y[:initial_window])))
def metrics(forecast):
    forecast = np.asarray(forecast)
    err = actual - forecast
    return {'MAE': np.mean(np.abs(err)), 'RMSE': np.sqrt(np.mean(err**2)), 'MASE': np.mean(np.abs(err)) / scale}
pd.DataFrame({'Naive': metrics(naive), 'AR(1)': metrics(ar1)}).T


The evaluation refits at each historical forecast origin. Nothing after the origin is allowed into model fitting or preprocessing.
